In [17]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/googleplaystore.csv")
print(df.shape)
df.head()

(10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 2.2 MB


In [19]:
df.isna().sum().sort_values(ascending=False)

Rating            1474
Current Ver          8
Android Ver          3
Content Rating       1
Type                 1
Size                 0
Reviews              0
Category             0
App                  0
Price                0
Installs             0
Last Updated         0
Genres               0
dtype: int64

In [20]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate app names:", df["App"].duplicated().sum())

Duplicate rows: 483
Duplicate app names: 1181


In [21]:
for col in ["Installs", "Price", "Size", "Reviews", "Category", "Type"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(8))


--- Installs ---
Installs
1,000,000+     1579
10,000,000+    1252
100,000+       1169
10,000+        1054
1,000+          907
5,000,000+      752
100+            719
500,000+        539
Name: count, dtype: int64

--- Price ---
Price
0        10040
$0.99      148
$2.99      129
$1.99       73
$4.99       72
$3.99       63
$1.49       46
$5.99       30
Name: count, dtype: int64

--- Size ---
Size
Varies with device    1695
11M                    198
12M                    196
14M                    194
13M                    191
15M                    184
17M                    160
19M                    154
Name: count, dtype: int64

--- Reviews ---
Reviews
0    596
1    272
2    214
3    175
4    137
5    108
6     97
7     90
Name: count, dtype: int64

--- Category ---
Category
FAMILY             1972
GAME               1144
TOOLS               843
MEDICAL             463
BUSINESS            460
PRODUCTIVITY        424
PERSONALIZATION     392
COMMUNICATION       387
Name: count, dtyp

In [22]:
df[df["Category"] == "1.9"]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


In [23]:
df[df["Type"].isna()]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
9148,Command & Conquer: Rivals,FAMILY,NaN,0,Varies with device,0,NaN,0,Everyone 10+,Strategy,"June 28, 2018",Varies with device,Varies with device


In [24]:
df[df["Installs"] == "0"]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
9148,Command & Conquer: Rivals,FAMILY,NaN,0,Varies with device,0,NaN,0,Everyone 10+,Strategy,"June 28, 2018",Varies with device,Varies with device


## Data Issues Found

**Dataset:** 10,841 rows × 13 columns. 12 columns are stored as text, and only `Rating` is numeric (float64).

### 1. Numbers stored as text (wrong data types)
- `Installs` holds text like "1,000,000+" and "10,000+". Remove "+" and "," and convert to integer. Values are bucket floors, not exact counts.
- `Price` holds text like "$0.99". 10,040 apps have "0" and the rest have a "$" prefix. Remove "$" and convert to float.
- `Reviews` is text. Convert to integer.
- `Last Updated` is text like "January 7, 2018". Convert to datetime.

### 2. Inconsistent `Size` column
- Values mix "19M" and "201k" formats.
- 1,695 apps (about 16%) say "Varies with device".
- Plan: convert to MB (k ÷ 1024) and leave "Varies with device" as missing. Do not fill it with a guess.

### 3. Missing values
| Column | Missing |
|---|---|
| Rating | 1,474 (about 13.6%) |
| Current Ver | 8 |
| Android Ver | 3 |
| Content Rating | 1 |
| Type | 1 |

- `Rating` is the biggest gap. Plan: do not impute it blindly. Exclude missing ratings only from rating-based analysis, and check whether they belong to apps with few or zero reviews (596 apps have 0 reviews).
- The remaining gaps are tiny (at most 8 rows), so they are safe to drop or ignore.
- The single missing `Type` is not a data-entry accident. It belongs to "Command & Conquer: Rivals" (row 9148), which has `Installs` = "0", 0 reviews, no rating and "Varies with device" in Size, Current Ver and Android Ver. It looks like an unreleased or pre-registration app. Plan: drop it, since it has no install history to analyse.

### 4. Duplicates
- 483 fully duplicated rows.
- 1,181 duplicate app names, so 698 repeated names have rows that differ (probably different review counts or versions).
- The same app counted more than once would inflate category totals and bias the results.
- Plan: keep one row per app, the row with the highest review count.

### 5. Corrupted row (index 10472)
- App: "Life Made WI-Fi Touchscreen Photo Frame". `Category` = "1.9", `Rating` = 19.0 (impossible; the scale is 1 to 5).
- Its values are shifted one column to the left: the row has no `Category` value, so everything after `App` sits one column too early. This puts "3.0M" in `Reviews`, "1,000+" in `Size`, "Free" in `Installs`, "Everyone" in `Price` and "0" in `Type`.
- This row would break the numeric conversions (for example, `int("Free")` fails).
- Plan: remove this row **first**, before any type conversion.

### Data limitations to state in the README
- Snapshot data (about 2018), so there is no time series and "growth" must be measured with proxies.
- Installs are ranges ("1,000,000+"), so the target `Success = Installs >= 1M` is a category, not an exact count.
- Analysis shows association, not causation.

### Cleaning plan (Step 2)
1. Drop the corrupted row (`Category == "1.9"`).
2. Convert `Installs`, `Price`, `Reviews`, `Last Updated` and `Size` to proper types.
3. Remove duplicates, keeping the highest-review row per app.
4. Handle missing values as described above.
5. Engineer features: `Success`, `Review_Rate`, `Days_Since_Update`, `Price_Bucket`.

In [25]:
df[df["Installs"] == "0"]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
9148,Command & Conquer: Rivals,FAMILY,NaN,0,Varies with device,0,NaN,0,Everyone 10+,Strategy,"June 28, 2018",Varies with device,Varies with device


In [26]:
df_clean = df.copy()
print("Start:", df_clean.shape)

df_clean = df_clean[df_clean["Category"] != "1.9"]
print("After dropping corrupted row:", df_clean.shape)

Start: (10841, 13)
After dropping corrupted row: (10840, 13)


In [27]:
df_clean = df_clean[df_clean["Installs"] != "0"]
print("After dropping unreleased apps:", df_clean.shape)

After dropping unreleased apps: (10839, 13)


In [28]:
# Installs: "1,000,000+" -> 1000000
df_clean["Installs"] = df_clean["Installs"].str.replace("[+,]", "", regex=True).astype(int)

# Price: "$0.99" -> 0.99
df_clean["Price"] = df_clean["Price"].str.replace("$", "", regex=False).astype(float)

# Reviews: text -> integer
df_clean["Reviews"] = df_clean["Reviews"].astype(int)

# Last Updated: text -> date
df_clean["Last Updated"] = pd.to_datetime(df_clean["Last Updated"])

# Size: "19M" -> 19.0, "201k" -> 0.196, "Varies with device" -> missing
def size_to_mb(s):
    if s.endswith("M"):
        return float(s[:-1])
    if s.endswith("k"):
        return float(s[:-1]) / 1024
    return np.nan

df_clean["Size_MB"] = df_clean["Size"].apply(size_to_mb)

In [29]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 10839 entries, 0 to 10840
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             10839 non-null  str           
 1   Category        10839 non-null  str           
 2   Rating          9366 non-null   float64       
 3   Reviews         10839 non-null  int64         
 4   Size            10839 non-null  str           
 5   Installs        10839 non-null  int64         
 6   Type            10839 non-null  str           
 7   Price           10839 non-null  float64       
 8   Content Rating  10839 non-null  str           
 9   Genres          10839 non-null  str           
 10  Last Updated    10839 non-null  datetime64[us]
 11  Current Ver     10831 non-null  str           
 12  Android Ver     10837 non-null  str           
 13  Size_MB         9145 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(8)
memory usage: 2.1 M

In [30]:
df_clean = (df_clean
            .sort_values("Reviews", ascending=False)
            .drop_duplicates(subset="App", keep="first")
            .reset_index(drop=True))

print("After removing duplicates:", df_clean.shape)
print("Duplicate app names left:", df_clean["App"].duplicated().sum())

After removing duplicates: (9658, 14)
Duplicate app names left: 0


In [31]:
print(df_clean.isna().sum())
df_clean[["Rating", "Reviews", "Installs", "Price", "Size_MB"]].describe()

App                  0
Category             0
Rating            1462
Reviews              0
Size                 0
Installs             0
Type                 0
Price                0
Content Rating       0
Genres               0
Last Updated         0
Current Ver          8
Android Ver          2
Size_MB           1227
dtype: int64


,Rating,Reviews,Installs,Price,Size_MB
count,8196.000000,9.658000e+03,9.658000e+03,9658.000000,8431.000000
mean,4.173267,2.168266e+05,7.798978e+06,1.097344,20.398075
std,0.536253,1.831524e+06,5.377245e+07,16.852487,21.828959
min,1.000000,0.000000e+00,0.000000e+00,0.000000,0.008301
25%,4.000000,2.500000e+01,1.000000e+03,0.000000,4.600000
50%,4.300000,9.695000e+02,1.000000e+05,0.000000,12.000000
75%,4.500000,2.945775e+04,1.000000e+06,0.000000,28.000000
max,5.000000,7.815831e+07,1.000000e+09,400.000000,100.000000


In [32]:
df_clean.to_csv("../data/cleaned/playstore_clean.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (9658, 14)


## Cleaning Log
- Raw: 10,841 rows
- Dropped corrupted row (Category = "1.9"): 10,840
- Dropped unreleased apps (Installs = "0"): (10839, 13)
- Converted Installs, Price, Reviews, Last Updated, Size to proper types
- Removed duplicate apps (kept highest review count): __0_ final rows
- Rating left with missing values (excluded only from rating analysis)

In [33]:
df_clean = df.copy()
print("Raw:", df_clean.shape)

# 1. Drop the corrupted row
df_clean = df_clean[df_clean["Category"] != "1.9"]
print("After dropping corrupted row:", df_clean.shape)

# 2. Drop unreleased apps
df_clean = df_clean[df_clean["Installs"] != "0"]
print("After dropping unreleased apps:", df_clean.shape)

# 3. Convert data types
df_clean["Installs"] = df_clean["Installs"].str.replace("[+,]", "", regex=True).astype(int)
df_clean["Price"] = df_clean["Price"].str.replace("$", "", regex=False).astype(float)
df_clean["Reviews"] = df_clean["Reviews"].astype(int)
df_clean["Last Updated"] = pd.to_datetime(df_clean["Last Updated"])

def size_to_mb(s):
    if s.endswith("M"):
        return float(s[:-1])
    if s.endswith("k"):
        return float(s[:-1]) / 1024
    return np.nan

df_clean["Size_MB"] = df_clean["Size"].apply(size_to_mb)

# 4. Remove duplicate apps (keep the row with the most reviews)
df_clean = (df_clean
            .sort_values("Reviews", ascending=False)
            .drop_duplicates(subset="App", keep="first")
            .reset_index(drop=True))
print("After removing duplicates:", df_clean.shape)
print("Duplicate app names left:", df_clean["App"].duplicated().sum())

# 5. Save
df_clean.to_csv("../data/cleaned/playstore_clean.csv", index=False)
print("Saved.")

Raw: (10841, 13)
After dropping corrupted row: (10840, 13)
After dropping unreleased apps: (10839, 13)
After removing duplicates: (9658, 14)
Duplicate app names left: 0
Saved.


In [34]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 9658 entries, 0 to 9657
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             9658 non-null   str           
 1   Category        9658 non-null   str           
 2   Rating          8196 non-null   float64       
 3   Reviews         9658 non-null   int64         
 4   Size            9658 non-null   str           
 5   Installs        9658 non-null   int64         
 6   Type            9658 non-null   str           
 7   Price           9658 non-null   float64       
 8   Content Rating  9658 non-null   str           
 9   Genres          9658 non-null   str           
 10  Last Updated    9658 non-null   datetime64[us]
 11  Current Ver     9650 non-null   str           
 12  Android Ver     9656 non-null   str           
 13  Size_MB         8431 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(8)
memory usage: 1.

In [35]:
df_clean.isna().sum().sort_values(ascending=False)

Rating            1462
Size_MB           1227
Current Ver          8
Android Ver          2
Category             0
App                  0
Installs             0
Size                 0
Reviews              0
Type                 0
Genres               0
Content Rating       0
Price                0
Last Updated         0
dtype: int64

In [36]:
df_clean[["Rating", "Reviews", "Installs", "Price", "Size_MB"]].describe()

,Rating,Reviews,Installs,Price,Size_MB
count,8196.000000,9.658000e+03,9.658000e+03,9658.000000,8431.000000
mean,4.173267,2.168266e+05,7.798978e+06,1.097344,20.398075
std,0.536253,1.831524e+06,5.377245e+07,16.852487,21.828959
min,1.000000,0.000000e+00,0.000000e+00,0.000000,0.008301
25%,4.000000,2.500000e+01,1.000000e+03,0.000000,4.600000
50%,4.300000,9.695000e+02,1.000000e+05,0.000000,12.000000
75%,4.500000,2.945775e+04,1.000000e+06,0.000000,28.000000
max,5.000000,7.815831e+07,1.000000e+09,400.000000,100.000000


In [37]:
# Are missing ratings mostly apps that few people use?
df_clean.groupby(df_clean["Rating"].isna())["Reviews"].median()

Rating
False    3017.0
True        1.0
Name: Reviews, dtype: float64

## Cleaning Log

| Step | Rows after |
|---|---|
| Raw data | 10,841 |
| Dropped corrupted row (Category = "1.9") | 10,840 |
| Dropped unreleased app (Installs = "0") | 10,839 |
| Removed 1,181 duplicate app names (kept highest review count) | **9,658** |

**Type conversions:** `Installs`, `Reviews` to integer; `Price` to float; `Last Updated` to datetime; `Size` to `Size_MB` (k ÷ 1024, "Varies with device" left as missing).

**Missing values left on purpose:**
- `Rating`: 1,462 (15.1%). Excluded only from rating-based analysis, not imputed.
- `Size_MB`: 1,227 (12.7%, "Varies with device"). Excluded only from size analysis.
- `Current Ver` (8) and `Android Ver` (2) are not used in the analysis.

**Result:** 9,658 unique apps, saved to `data/cleaned/playstore_clean.csv`.

---

## Quick Data Checks (after cleaning)

| Column | Count | Mean | Median | Min | Max |
|---|---|---|---|---|---|
| Rating | 8,196 | 4.17 | 4.3 | 1.0 | 5.0 |
| Reviews | 9,658 | 216,827 | 970 | 0 | 78,158,310 |
| Installs | 9,658 | 7,798,978 | 100,000 | 0 | 1,000,000,000 |
| Price ($) | 9,658 | 1.10 | 0 | 0 | 400 |
| Size_MB | 8,431 | 20.4 | 12.0 | 0.01 | 100 |

**What the checks show:**
- **The corrupted row is gone.** `Rating` now stays between 1.0 and 5.0 (it was 19.0 before).
- **Installs and Reviews are extremely right-skewed.** The mean installs (about 7.8M) is 78 times the median (100K), and the mean reviews (about 217K) is over 200 times the median (970). A few giant apps pull the mean up. **Plan:** use medians, log scales and non-parametric tests, not means.
- **Most apps are free.** The median and 75th percentile of `Price` are both $0, so the $1.10 mean is driven by paid apps. The $400 maximum is an outlier (probably a novelty app like "I am rich"), so treat prices carefully.
- **- **The success threshold is meaningful.** 35.2% of apps (3,398 of 9,658) reach 1M+ installs, so it is a common outcome in this dataset, not a rare one..
- **Some apps have 0 installs and 0 reviews** (both minimums are 0). Number of apps with `Installs == 0` (the "0+" bucket): ___ *(fill in from the Step 3 Cell 1 check)*. Plan: `Review_Rate` treats them as missing to avoid dividing by zero.

**Are missing ratings random?** No. Apps with a missing `Rating` have a **median of 1 review**, versus **3,017 reviews** for apps with a rating. Missing ratings are almost entirely apps with almost no users, so they cannot be treated as random gaps. Decision: exclude them from rating-based analysis and do not impute them, since a made-up rating for an app nobody reviewed would add false information.

## Step 3: Feature Engineering

In [38]:
print("Apps with 0 installs:", (df_clean["Installs"] == 0).sum())
print("Apps with 0 reviews:", (df_clean["Reviews"] == 0).sum())

Apps with 0 installs: 14
Apps with 0 reviews: 592


In [39]:
df_clean["Success"] = (df_clean["Installs"] >= 1_000_000).astype(int)

print(df_clean["Success"].value_counts())
print(df_clean["Success"].value_counts(normalize=True).round(3))

Success
0    6260
1    3398
Name: count, dtype: int64
Success
0    0.648
1    0.352
Name: proportion, dtype: float64


In [40]:
df_clean["Review_Rate"] = df_clean["Reviews"] / df_clean["Installs"].replace(0, np.nan)

print(df_clean["Review_Rate"].describe())
print("Apps with Review_Rate > 1:", (df_clean["Review_Rate"] > 1).sum())

count    9644.000000
mean        0.038117
std         0.100057
min         0.000000
25%         0.007386
50%         0.017000
75%         0.038162
max         4.000000
Name: Review_Rate, dtype: float64
Apps with Review_Rate > 1: 11


In [41]:
latest = df_clean["Last Updated"].max()
print("Newest update date in the dataset:", latest)

df_clean["Days_Since_Update"] = (latest - df_clean["Last Updated"]).dt.days

df_clean["Days_Since_Update"].describe()

Newest update date in the dataset: 2018-08-08 00:00:00


count    9658.00000
mean      281.03510
std       406.73631
min         0.00000
25%        22.00000
50%        96.00000
75%       366.00000
max      3001.00000
Name: Days_Since_Update, dtype: float64

In [42]:
df_clean["Install_Tier"] = pd.cut(
    df_clean["Installs"],
    bins=[0, 1_000, 100_000, 1_000_000, 10_000_000, np.inf],
    labels=["<1K", "1K-100K", "100K-1M", "1M-10M", "10M+"],
    right=False   # lower edge included: 1,000,000 falls in "1M-10M"
)

df_clean["Install_Tier"].value_counts().sort_index()

Install_Tier
<1K        1789
1K-100K    2854
100K-1M    1617
1M-10M     2024
10M+       1374
Name: count, dtype: int64

In [43]:
df_clean["Price_Bucket"] = pd.cut(
    df_clean["Price"],
    bins=[-1, 0, 2, 5, 10, 1000],
    labels=["Free", "$0.01-2", "$2-5", "$5-10", "$10+"]
)

df_clean["Price_Bucket"].value_counts().sort_index()

Price_Bucket
Free       8904
$0.01-2     283
$2-5        314
$5-10        84
$10+         73
Name: count, dtype: int64

In [44]:
# H4: app size
df_clean["Size_Bucket"] = pd.cut(
    df_clean["Size_MB"],
    bins=[0, 5, 15, 30, 60, 100],
    labels=["<5MB", "5-15MB", "15-30MB", "30-60MB", "60-100MB"]
)

# H3: update recency
df_clean["Update_Bucket"] = pd.cut(
    df_clean["Days_Since_Update"],
    bins=[-1, 30, 90, 180, 365, 100_000],
    labels=["<=30d", "31-90d", "91-180d", "181-365d", ">1yr"]
)

# Log scale of installs, for charts and correlations (installs are heavily skewed)
df_clean["Log_Installs"] = np.log10(df_clean["Installs"] + 1)

print(df_clean["Size_Bucket"].value_counts().sort_index())
print()
print(df_clean["Update_Bucket"].value_counts().sort_index())

Size_Bucket
<5MB        2317
5-15MB      2494
15-30MB     1719
30-60MB     1301
60-100MB     600
Name: count, dtype: int64

Update_Bucket
<=30d       2936
31-90d      1828
91-180d     1158
181-365d    1320
>1yr        2416
Name: count, dtype: int64


In [45]:
# 1. Success should exist only in the top two tiers
print(pd.crosstab(df_clean["Install_Tier"], df_clean["Success"]))

# 2. Shape and new columns
print(df_clean.shape)
print(df_clean.columns.tolist())

Success          0     1
Install_Tier            
<1K           1789     0
1K-100K       2854     0
100K-1M       1617     0
1M-10M           0  2024
10M+             0  1374
(9658, 22)
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver', 'Size_MB', 'Success', 'Review_Rate', 'Days_Since_Update', 'Install_Tier', 'Price_Bucket', 'Size_Bucket', 'Update_Bucket', 'Log_Installs']


In [46]:
df_clean.to_csv("../data/cleaned/playstore_features.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (9658, 22)


## Feature Dictionary

| Feature | Definition | Why it's needed |
| sees the feature and installs and review rate of the apps |
| `Success` | 1 if Installs ≥ 1,000,000, else 0 | Target variable for hypothesis tests and ML. Success rate: __35.2_% |
| `Review_Rate` | Reviews ÷ Installs (NaN when Installs = 0) | Engagement proxy. Median: __50%_ |
| `Days_Since_Update` | Days between the app's last update and the newest update in the dataset ({latest date}) | Update recency (H3) |
| `Install_Tier` | <1K, 1K-100K, 100K-1M, 1M-10M, 10M+ | Readable grouping for charts |
| `Price_Bucket` | Free, $0.01-2, $2-5, $5-10, $10+ | Pricing analysis (H2) |
| `Size_Bucket` | <5MB to 60-100MB | Size analysis (H4) |
| `Update_Bucket` | ≤30d to >1yr | Update analysis (H3) |
| `Log_Installs` | log10(Installs + 1) | Reduces skew for charts and correlations |

**Notes:**
- Apps with 0 installs: _14_ (Review_Rate left as missing for these).
- Apps with Review_Rate > 1: _11_ (an artifact of Installs being a bucket floor).
- Recency is measured against the dataset's newest date, not today.

In [47]:
print("Newest update date:", df_clean["Last Updated"].max())
print("Median Review_Rate:", round(df_clean["Review_Rate"].median(), 4))
print("Apps with Review_Rate > 1:", (df_clean["Review_Rate"] > 1).sum())
print()
print(df_clean["Size_Bucket"].value_counts().sort_index())
print()
print(df_clean["Update_Bucket"].value_counts().sort_index())

Newest update date: 2018-08-08 00:00:00
Median Review_Rate: 0.017
Apps with Review_Rate > 1: 11

Size_Bucket
<5MB        2317
5-15MB      2494
15-30MB     1719
30-60MB     1301
60-100MB     600
Name: count, dtype: int64

Update_Bucket
<=30d       2936
31-90d      1828
91-180d     1158
181-365d    1320
>1yr        2416
Name: count, dtype: int64
